# Schema and Data Quality Analysis

## Purpose
This notebook establishes a clear understanding of the dataset schema and assesses
data quality before any exploratory or modeling work.

## Scope
- Identify column roles (feature, target, identifier)
- Validate data types and ranges
- Detect missing values, duplicates, and integrity issues
- Flag early risks such as data leakage or bias

## Notes
- Dataset is synthetic and used for crime analytics research only.
- No real-world identification or surveillance use.


In [3]:
# Importing libraries

import os
import pandas as pd
import numpy as np


# Configurations
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 120)

RANDOM_STATE = 42

In [4]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
data_path = os.path.join(project_root,'data', 'interim', 'cleaned_data.csv')

crime_data = pd.read_csv(data_path)
crime_data.shape

(10000, 10)

In [5]:
crime_data.head()

,crime_id,crime_type,latitude,longitude,hour,day_of_week,victim_age,suspect_age,weapon_used,arrest_made
0,1,Homicide,87.107569,0.435870,12,Fri,30,53,gun,0
1,2,Burglary,73.104757,71.107880,21,Fri,15,56,unknown,0
2,3,Fraud,-80.019858,-153.780218,7,Wed,39,64,blunt object,0
3,4,Homicide,59.946168,157.935894,20,Thu,32,54,gun,1
4,5,Robbery,-7.595112,-14.848008,10,Mon,45,69,unknown,0


In [6]:
crime_data.tail()

,crime_id,crime_type,latitude,longitude,hour,day_of_week,victim_age,suspect_age,weapon_used,arrest_made
9995,9996,Robbery,15.615472,-48.550187,18,Sun,29,39,blunt object,0
9996,9997,Fraud,18.905574,0.306350,8,Sun,16,60,gun,0
9997,9998,Assault,-68.068960,179.605212,14,Tue,61,51,unknown,0
9998,9999,Robbery,8.753960,6.755842,6,Mon,46,21,blunt object,1
9999,10000,Assault,-87.851106,167.112535,5,Tue,55,31,blunt object,0


In [7]:
crime_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   crime_id     10000 non-null  int64  
 1   crime_type   10000 non-null  object 
 2   latitude     10000 non-null  float64
 3   longitude    10000 non-null  float64
 4   hour         10000 non-null  int64  
 5   day_of_week  10000 non-null  object 
 6   victim_age   10000 non-null  int64  
 7   suspect_age  10000 non-null  int64  
 8   weapon_used  10000 non-null  object 
 9   arrest_made  10000 non-null  int64  
dtypes: float64(2), int64(5), object(3)
memory usage: 781.4+ KB


In [8]:
missing_counts = crime_data.isna().sum()
missing_percent = (missing_counts/len(crime_data)) * 100

pd.DataFrame({
    "missing_counts": missing_counts,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

,missing_counts,missing_percent
crime_id,0,0.0
crime_type,0,0.0
latitude,0,0.0
longitude,0,0.0
hour,0,0.0
day_of_week,0,0.0
victim_age,0,0.0
suspect_age,0,0.0
weapon_used,0,0.0
arrest_made,0,0.0


## Column Roles (Post-Cleaning)

| Column Name   | Role        | Notes |
|--------------|-------------|------|
| crime_id     | Identifier  | Excluded from modeling |
| crime_type   | Target      | Multi-class |
| weapon_used  | Feature     | Includes 'unknown' |
| latitude     | Feature     | Requires binning |
| longitude    | Feature     | Requires binning |
| hour         | Feature     | Cyclical |
| day_of_week  | Feature     | Categorical |


In [9]:
crime_data.duplicated().sum()

np.int64(0)

In [10]:
crime_data["crime_id"].nunique(), len(crime_data)

(10000, 10000)

In [11]:
assert crime_data["hour"].between(0, 23).all()
assert crime_data["latitude"].between(-90, 90).all()
assert crime_data["longitude"].between(-180, 180).all()

In [12]:
crime_data["weapon_used"].value_counts()

weapon_used
blunt object    2552
knife           2533
unknown         2475
gun             2440
Name: count, dtype: int64

In [13]:
crime_data["crime_type"].value_counts(normalize=True)

crime_type
Fraud        0.1464
Burglary     0.1439
Homicide     0.1429
Assault      0.1428
Vandalism    0.1426
Theft        0.1423
Robbery      0.1391
Name: proportion, dtype: float64

## Leakage Review (Post-Cleaning)

- No features were derived using target information
- Missing categorical values handled semantically
- `weapon_used = "unknown"` does not encode outcome data
- No temporal leakage introduced

Status: LOW RISK


## Conclusion

The interim dataset satisfies schema and quality requirements:
- Consistent column names
- Explicit handling of missing categorical data
- Valid numeric ranges
- No duplicates or identifier corruption

The dataset is approved for:
- Univariate analysis
- Feature engineering
- Validation pipelines
